In [1]:
import pandas as pd

### Load the data

In [264]:
cubo = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\cubo_1.xlsx", 
                    sheet_name = 'SKUs',
                    skiprows=4
                    )

In [30]:
ventas1 = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\ventasxfact_08_10.csv")

In [29]:
ventas2 = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\ventasxfact_11_12.csv")

In [31]:
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdgs.xlsx")

In [104]:
ventasxfact =pd.concat([ventas1, ventas2], ignore_index=True)

In [105]:
ventasxfact.shape

(18430295, 6)

In [106]:
ventasxfact['FECHA']= pd.to_datetime(ventasxfact['FECHA'])

In [107]:
ventasxfact = ventasxfact[ventasxfact['FECHA'] > '2025-09-30 00:00:00']

### Procesamos la data

In [265]:
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact['COD_ARTICULO'] = ventasxfact['COD_ARTICULO'].astype(str)

In [266]:
# filtering only SKUs on FARMACORP
ventas_farmacia = pd.merge(cubo,
                            ventasxfact,
                            on='COD_ARTICULO',
                            how='inner'
                            )

In [267]:
#vamos a filtrar solo BODEGAS FARMACORP para este analisis
clusters_fc = clusters[clusters['UNE']=='FARMACORP']

In [268]:
#base sobre la cual se va a trabajar
ventas_farmacia = pd.merge(
    ventas_farmacia,
    clusters_fc,
    left_on='COD_BODEGA',
    right_on='BODEGA',
)

In [269]:
ventas_farmacia.columns

Index(['CAT 1', 'CAT 2', 'CAT 3', 'CAT 4', 'COD_ARTICULO', 'ARTICULO',
       'Precio Unitario FA', 'CR Unitario', 'FECHA', 'COD_BODEGA',
       'NRO_FACTURAS', '# Sell-OUT', 'Ventas', 'BODEGA', 'UNE', 'BODEGA2',
       'CIUDAD', 'REGION', 'FORMATO', 'NSE', 'CLUSTER', 'BEAUTY',
       'HOSPITALARIA'],
      dtype='object')

In [270]:
ventas_farmacia['CAT 1'].unique()

array(['ABARROTES', 'AGUAS-ISOTONICOS Y ENERGIZANTES',
       'ANTIARLEGICOS Y RESPIRATORIOS', 'CARNES',
       'COLIRIOS (OFTAMOLOGICO)', 'COMIDAS PREPARADAS',
       'CUIDADO DEL HOGAR', 'CUIDADO INFANTIL', 'CUIDADO PERSONAL VARIOS',
       'DERMATOLOGICOS RECETADOS', 'ELECTROHOGAR', 'ETICOS AGUDOS',
       'ETICOS CRONICOS', 'ETICOS TRATAMIENTO',
       'FIAMBRES-EMBUTIDOS Y CONGELADOS', 'FORMULAS INFANTILES',
       'FRUTAS Y VERDURAS', 'GALLETAS-CHOCOLATES Y SNACKS',
       'GASEOSAS Y JUGOS', 'GASTRICO', 'HELADOS-PALETAS Y HIELO',
       'HIGIENE PERSONAL', 'HOGAR Y VARIOS', 'INSUMOS MEDICOS',
       'LACTEOS Y DERIVADOS', 'NUTRICION INFANTIL', 'PANADERIA ENVASADA',
       'PANALES Y TOALLITAS HUMEDAS', 'PRODUCTOS DE CANJE',
       'RESFRIO/DOLOR', 'SALUD Y VARIOS OTC', 'SKIN CARE Y MAQUILLAJE',
       'SUEROS', 'VACUNAS', 'VITAMINAS Y MINERALES'], dtype=object)

In [271]:
skincare = ventas_farmacia[ventas_farmacia['CAT 2']=='SKIN CARE']
skincare.NRO_FACTURAS.nunique()

57623

In [272]:
maquillaje = ventas_farmacia[ventas_farmacia['CAT 2']=='MAQUILLAJE']
maquillaje.NRO_FACTURAS.nunique()

64450

In [273]:
tratamiento = ventas_farmacia[ventas_farmacia['CAT 1']=='ETICOS TRATAMIENTO']
tratamiento.NRO_FACTURAS.nunique()

556426

### Calculamos medidas

In [274]:
#calculamos el total de facturas emitidas en el rango de fechas de los datos
total_fact = ventas_farmacia['NRO_FACTURAS'].nunique()
total_fact

4964955

In [275]:
#contamos facturas en las que aparece al menos 1 vez cada SKU
ventas_sku = ventas_farmacia.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NRO_FACTURAS'].nunique().reset_index().sort_values(by='NRO_FACTURAS', ascending=False)

In [276]:
#calculamos el peso de cada SKU sobre el total de las facturas
ventas_sku['weight'] = (ventas_sku['NRO_FACTURAS']/total_fact).round(8)

In [277]:
#seleccionamos las facturas que solo tienen 1 sku
facturas_unicas = (
    ventas_farmacia
    .groupby(['NRO_FACTURAS'])
    .size()
    .loc[lambda x: x == 1]
    .reset_index()[['NRO_FACTURAS']]
)

#filtramos la data con los sku que aparecen solos en 1 factura
ventas_solo = ventas_farmacia.merge(
    facturas_unicas,
    on=['NRO_FACTURAS'],
    how='inner'
)

In [278]:
cronicos_solos = ventas_solo[ventas_solo['CAT 1']=='ETICOS CRONICOS']
cronicos_solos.NRO_FACTURAS.nunique()

221911

In [279]:
skincare_solos = ventas_solo[ventas_solo['CAT 2']=='SKIN CARE']
skincare_solos.NRO_FACTURAS.nunique()

18977

In [280]:
skincare_solos

,CAT 1,CAT 2,CAT 3,CAT 4,COD_ARTICULO,ARTICULO,Precio Unitario FA,CR Unitario,FECHA,COD_BODEGA,...,BODEGA,UNE,BODEGA2,CIUDAD,REGION,FORMATO,NSE,CLUSTER,BEAUTY,HOSPITALARIA
2590856,SKIN CARE Y MAQUILLAJE,SKIN CARE,ACCESORIOS DE BELLEZA,ACCESORIOS DE BELLEZA,06938294612163,QVS/TRUYU URBAN SET MINI JUEGO D/PINZAS URBANA...,39.0,16.232575,2025-10-01,B118,...,B118,FARMACORP,SC-25 SN.MARTIN #335-EQUIPETROL,SANTA CRUZ,LLANO,MEDIANA,A,MEDIANA-A,SI,SI
2590857,SKIN CARE Y MAQUILLAJE,SKIN CARE,ACCESORIOS DE BELLEZA,ACCESORIOS DE BELLEZA,06938294612163,QVS/TRUYU URBAN SET MINI JUEGO D/PINZAS URBANA...,39.0,16.232575,2025-10-01,B153,...,B153,FARMACORP,SC -73 MALL VENTURA,SANTA CRUZ,LLANO,MEDIANA,A,MEDIANA-A,SI,NO
2590858,SKIN CARE Y MAQUILLAJE,SKIN CARE,ACCESORIOS DE BELLEZA,ACCESORIOS DE BELLEZA,06938294612163,QVS/TRUYU URBAN SET MINI JUEGO D/PINZAS URBANA...,39.0,16.232575,2025-10-01,B154,...,B154,FARMACORP,SC-77 FCORP. AV. 25 DE MAYO #4 (WARNES),SANTA CRUZ,LLANO,MEDIANA,B,MEDIANA-B,SI,NO
2590859,SKIN CARE Y MAQUILLAJE,SKIN CARE,ACCESORIOS DE BELLEZA,ACCESORIOS DE BELLEZA,06938294612163,QVS/TRUYU URBAN SET MINI JUEGO D/PINZAS URBANA...,39.0,16.232575,2025-10-01,B502,...,B502,FARMACORP,OR-105 AV.VILLARROEL ESQ.TARAPACA,ORURO,ALTIPLANO,GRANDE,B,GRANDE-B,SI,NO
2590860,SKIN CARE Y MAQUILLAJE,SKIN CARE,ACCESORIOS DE BELLEZA,ACCESORIOS DE BELLEZA,06938294612163,QVS/TRUYU URBAN SET MINI JUEGO D/PINZAS URBANA...,39.0,16.232575,2025-10-02,B170,...,B170,FARMACORP,SC-121 FCORP. AV. 2DO ANILLO PARAGUA,SANTA CRUZ,LLANO,PERMITIDO,B,PERMITIDO-B,NO,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2609828,SKIN CARE Y MAQUILLAJE,SKIN CARE,SERUM FACIALES,SERUMS DESPIGMENTANTE,8470001653505,NEORETIN DISCROM CONTROL SERUM X 30ML BOOSTER ...,449.2,333.450000,2025-11-26,B303,...,B303,FARMACORP,LP-59 SANCHEZ L. Esq. P.SALAZAR,LA PAZ,ALTIPLANO,MEDIANA,B,MEDIANA-B,SI,NO
2609829,SKIN CARE Y MAQUILLAJE,SKIN CARE,SERUM FACIALES,SERUMS DESPIGMENTANTE,8470001653505,NEORETIN DISCROM CONTROL SERUM X 30ML BOOSTER ...,449.2,333.450000,2025-12-10,B328,...,B328,FARMACORP,"LP - AV. HERNANDO SILES NRO. 6288, ZONA OBRAJES",LA PAZ,ALTIPLANO,MEDIANA,B,MEDIANA-B,NO,NO
2609830,SKIN CARE Y MAQUILLAJE,SKIN CARE,SERUM FACIALES,SERUMS DESPIGMENTANTE,8470001653505,NEORETIN DISCROM CONTROL SERUM X 30ML BOOSTER ...,449.2,333.450000,2025-12-15,B311,...,B311,FARMACORP,LP-89 AV SAAVEDRA #625 ESQ VILLALOBOS (MIRAF),LA PAZ,ALTIPLANO,PEQUENA,B,PEQUENA-B,SI,SI
2609831,SKIN CARE Y MAQUILLAJE,SKIN CARE,SERUM FACIALES,SERUMS DESPIGMENTANTE,8470001653505,NEORETIN DISCROM CONTROL SERUM X 30ML BOOSTER ...,449.2,333.450000,2025-12-24,B306,...,B306,FARMACORP,LP-79 C/17 A #864 Cdad.SATELITE(EL ALTO),LA PAZ,ALTIPLANO,MEDIANA,B,MEDIANA-B,SI,SI


In [281]:
maquillaje.ARTICULO.value_counts().head(30)

ARTICULO
SEPTONA DISCOS DE ALGODON X 100 UNID                     1479
FARMAX QUITA ESMALTE X 100ML C/ACETONA                   1176
NIVEA PROTECTOR BALSAMO LABIAL CHERRY SHINE              1149
FARMAX REMOVEDOR D/ESMALTE X 100ML SIN ACETONA            960
UNIK QUITAESMALTE X 110ML HUMECTANTE                      873
MANTEQUILLA DE CACAO BALSAMO LABIAL X 50 BARRAS           819
VOGUE MASCARA-PESTANA EFECTO TOTAL 6 NEGRO 9G             712
DR SELBY PROTECTOR LABIAL FPS15 X 5.7ML                   704
VOGUE MASCARA RESIST AGUA 9G                              698
PACK SEPTONA DAILY CLEAN DISCOS ALGODON 2+1               622
ASEPXIA BB MAQUILLAJE POLVO X10GR BEIGE CLARO MATE        615
ESSENCE JUICY BOMB SHINY LIPGLOSS 104 (10ML)              603
VOGUE MASCARA AMOR A PRIMERA VISTA WP 9GR                 587
LIPSTICK PROT LABIAL SPF30 UNID X 5.3GR SABOR SURTIDO     574
NIVEA PROTECTOR BALSAMO LABIAL FPS15 MED REPAIR           569
VOGUE BALSAMO LABIAL X 4.8GR CEREZA                       562

In [282]:
agudos_solos = ventas_solo[ventas_solo['CAT 1']=='ETICOS AGUDOS']
agudos_solos.NRO_FACTURAS.nunique()

535734

In [283]:
tratamiento_solos = ventas_solo[ventas_solo['CAT 1']=='ETICOS TRATAMIENTO']
tratamiento_solos.NRO_FACTURAS.nunique()

220163

In [284]:
#contamos facturas con un solo sku
total_fact_solo = ventas_solo['NRO_FACTURAS'].nunique()
total_fact_solo

2725763

In [285]:
#calculamos el % que representa del total de facturas
fact_solo = total_fact_solo/total_fact
fact_solo

0.5490005448186338

In [286]:
#contamos cuantas veces aparece cada sku solo 
ventas_sku_solo = ventas_solo.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NRO_FACTURAS'].count().reset_index().sort_values(by='NRO_FACTURAS', ascending=False)

In [287]:
ventas_sku_solo

,COD_ARTICULO,ARTICULO,NRO_FACTURAS
8008,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,29887
7123,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),26214
2144,255701,MIGRANOL X 100 COMP V+,19807
8626,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),14501
4410,700607,COLECTOR DE ORINA TRANS X 120/100/80ML STERIL ...,14249
...,...,...,...
14859,MQH-0154,SONDA DE ALIMENTACION ENTERAL AD 12FR SILMAG(K...,1
6,007806500168102,ELITE PANUELOS FACIAL TH CJA X 60UNID<DESC>,1
3288,4031571070693,ZOLEDRO DENK 1 AMP 4MG/5ML AC. ZOLEDRONICO <ONC>,1
3289,4031571073847,IMMUN ACTIVE DENK X 20 SOBRES,1


El SKU que se vende mas solo es Quetorol, seguido de Zopiclona, Migranol y Typirec

In [335]:
#calculamos el peso de cada sku en las facturas solas
ventas_sku_solo['weight'] = ventas_sku_solo['NRO_FACTURAS']/total_fact_solo

In [336]:
#unimos todo
skus_analysis = pd.merge(
    ventas_sku,
    ventas_sku_solo,
    on='COD_ARTICULO',
    how='left',
    suffixes=['_gral', '_alone']
)

In [337]:
#calculamos el % en peso de las veces que cada sku aparece solo vs el total de veces que aparece en una factura
skus_analysis['weight_overeach'] = skus_analysis['NRO_FACTURAS_alone']/skus_analysis['NRO_FACTURAS_gral']

In [338]:
skus_analysis['weight_overall'] = skus_analysis['NRO_FACTURAS_alone']/total_fact

In [339]:
skus_analysis.columns

Index(['COD_ARTICULO', 'ARTICULO_gral', 'NRO_FACTURAS_gral', 'weight_gral',
       'ARTICULO_alone', 'NRO_FACTURAS_alone', 'weight_alone',
       'weight_overeach', 'weight_overall'],
      dtype='object')

In [340]:
skus_analysis = skus_analysis[[
    'COD_ARTICULO', 'ARTICULO_gral', 
    'NRO_FACTURAS_gral', 'NRO_FACTURAS_alone', 
    'weight_gral','weight_alone',
    'weight_overeach', 'weight_overall'
]].copy()

NRO_FACTURAS_gral: número total de facturas en las que el SKU aparece al menos una vez, independientemente de si la factura contiene otros SKUs.

NRO_FACTURAS_alone: número de facturas en las que el SKU aparece de forma exclusiva, es decir, facturas que contienen únicamente ese SKU y ningún otro.

weight_gral: proporción de facturas en las que aparece el SKU respecto al total de facturas emitidas. Mide la presencia general del SKU en el conjunto completo de facturación.

weight_alone: proporción de facturas de un solo SKU en las que aparece el SKU, respecto al total de facturas que contienen únicamente un SKU. Refleja la participación del SKU dentro de las facturas unitarias.

weight_overeach: proporción de facturas en las que el SKU aparece solo respecto al total de facturas en las que dicho SKU aparece al menos una vez. Indica qué tan frecuentemente el SKU se vende de manera exclusiva cuando está presente en una factura.

weight_overall: proporción de facturas en las que el SKU aparece solo respecto al total de facturas emitidas. Representa el peso absoluto de las ventas exclusivas del SKU sobre toda la facturación.

In [341]:
skus_analysis = skus_analysis[skus_analysis['NRO_FACTURAS_alone']>=1000]

In [342]:
skus_analysis = skus_analysis.sort_values(
    by=['weight_gral'], 
    ascending=[False]).round(5)

In [216]:
#skus_analysis.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\analysis_sku.xlsx", index=False)

In [343]:
ventas_farmacia.columns

Index(['CAT 1', 'CAT 2', 'CAT 3', 'CAT 4', 'COD_ARTICULO', 'ARTICULO',
       'Precio Unitario FA', 'CR Unitario', 'FECHA', 'COD_BODEGA',
       'NRO_FACTURAS', '# Sell-OUT', 'Ventas', 'BODEGA', 'UNE', 'BODEGA2',
       'CIUDAD', 'REGION', 'FORMATO', 'NSE', 'CLUSTER', 'BEAUTY',
       'HOSPITALARIA'],
      dtype='object')

### Searching the AyB best pairs for top 500

In [344]:
# selfmerge for finding pairs of CAT 4 on same invoice
pairs_factura = pd.merge(
    ventas_farmacia,
    ventas_farmacia,
    on=["NRO_FACTURAS"],
    suffixes=("_A", "_B")
)

KeyboardInterrupt: 

In [345]:
pairs_factura.columns

Index(['CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A',
       'ARTICULO_A', 'Precio Unitario FA_A', 'CR Unitario_A', 'FECHA_A',
       'COD_BODEGA_A', 'NRO_FACTURAS', '# Sell-OUT_A', 'Ventas_A', 'BODEGA_A',
       'UNE_A', 'BODEGA2_A', 'CIUDAD_A', 'REGION_A', 'FORMATO_A', 'NSE_A',
       'CLUSTER_A', 'BEAUTY_A', 'HOSPITALARIA_A', 'CAT 1_B', 'CAT 2_B',
       'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B',
       'Precio Unitario FA_B', 'CR Unitario_B', 'FECHA_B', 'COD_BODEGA_B',
       '# Sell-OUT_B', 'Ventas_B', 'BODEGA_B', 'UNE_B', 'BODEGA2_B',
       'CIUDAD_B', 'REGION_B', 'FORMATO_B', 'NSE_B', 'CLUSTER_B', 'BEAUTY_B',
       'HOSPITALARIA_B'],
      dtype='object')

In [346]:
# filtering combinations to avoid duplicates
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] != pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] > pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["CAT 4_A"] != pairs_factura["CAT 4_B"]]


In [347]:
ventasABxfac = pairs_factura.groupby([
    'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A',
    'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B'
]
    ).agg({'NRO_FACTURAS':'nunique'}).reset_index()

In [348]:
ventasABxfac = ventasABxfac.rename(columns={'NRO_FACTURAS' : 'NRO_FACTURAS_AB'})

In [349]:
ventasABxfac.sort_values(by=['NRO_FACTURAS_AB'], ascending=[False])

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,COD_ARTICULO_B,ARTICULO_B,NRO_FACTURAS_AB
2239792,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,106421,JERINGA DESC 10ML C/AG 21 X 1 1/2 FARMACORP<H...,ETICOS AGUDOS,HORMONAS,CORTICOSTEROIDES VIA GENERAL(CORTICOIDES),CORTICOSTEROIDES SOLOS,751140,DEXAMETASONA 8MG IM-IV X 100 AMP/2ML GENERICO LI,5005
3138631,VITAMINAS Y MINERALES,VITAMINAS,VITAMINA C,VITAMINA C,751304,VITAMINA C 1GR IV X 50 AMP 5ML/2ML GENERICO LI,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARTABLES HOSPITALARIOS,709726,EQUIPO P/SUERO EN Y VENOCLISI C/AGUJA DOBLE PU...,3554
2297313,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARTABLES HOSPITALARIOS,709726,EQUIPO P/SUERO EN Y VENOCLISI C/AGUJA DOBLE PU...,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,240141,BRANULA #24 3/4 <HP>,3379
2239642,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,106421,JERINGA DESC 10ML C/AG 21 X 1 1/2 FARMACORP<H...,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,751145,DICLOFENACO 75MG IM X 50 AMP GENERICO LI,3324
2297312,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARTABLES HOSPITALARIOS,709726,EQUIPO P/SUERO EN Y VENOCLISI C/AGUJA DOBLE PU...,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,240136,BRANULA #22 X 1 <HP>,3314
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,GASEOSAS Y JUGOS,JUGOS,JUGOS,JUGOS,7771609002926,ADES SOYA X 1LT MANZANA SIN GLUTEN,1
12,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,GALLETAS-CHOCOLATES Y SNACKS,SNACKS Y PIQUEOS,MANI,MANI,7770110100022,CLAF PASANKALLA X 170GR PORORO BANADO C/CHANCACA,1
11,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,GALLETAS-CHOCOLATES Y SNACKS,GALLETAS,OTRAS GALLETAS,OTRAS GALLETAS,07891962035451,BAUDUCCO GALLETAS CHAMPANHE X 150GR C/AZU CRIS...,1
10,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,GALLETAS-CHOCOLATES Y SNACKS,GALLETAS,GALLETAS DULCES,GALLETAS DULCES,7802215505409,COSTA COCO EXP X 125GR,1


In [350]:
ventasABxfac['weight_AB'] = ventasABxfac['NRO_FACTURAS_AB'] / total_fact

In [351]:
ventasxpairs = pd.merge(
    ventasABxfac,
    ventas_sku,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO'
)

In [352]:
ventasxpairs.columns

Index(['CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A',
       'ARTICULO_A', 'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B',
       'COD_ARTICULO_B', 'ARTICULO_B', 'NRO_FACTURAS_AB', 'weight_AB',
       'COD_ARTICULO', 'ARTICULO', 'NRO_FACTURAS', 'weight'],
      dtype='object')

In [353]:
ventasxpairs_final = pd.merge(
    ventasxpairs[[
        'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
        'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
        'NRO_FACTURAS_AB', 'weight_AB', 'NRO_FACTURAS', 'weight']],
    ventas_sku,
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO'
)

In [354]:
ventasxpairs_final = ventasxpairs_final.rename(columns={'weight_x':'weight_A',
                                                        'NRO_FACTURAS_x':'NRO_FACTURAS_A',
                                                        'weight_y':'weight_B',
                                                        'NRO_FACTURAS_y':'NRO_FACTURAS_B'})

In [355]:
ventasxpairs_final.columns

Index(['CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A',
       'ARTICULO_A', 'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B',
       'COD_ARTICULO_B', 'ARTICULO_B', 'NRO_FACTURAS_AB', 'weight_AB',
       'NRO_FACTURAS_A', 'weight_A', 'COD_ARTICULO', 'ARTICULO',
       'NRO_FACTURAS_B', 'weight_B'],
      dtype='object')

In [356]:
ventasxpairs_final = ventasxpairs_final[[
    'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
    'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
    'NRO_FACTURAS_A', 'weight_A', 
    'NRO_FACTURAS_B', 'weight_B',
    'NRO_FACTURAS_AB', 'weight_AB'
]]

In [357]:
ventasxpairs_final = ventasxpairs_final.sort_values(
    by=['COD_ARTICULO_A',
        'NRO_FACTURAS_AB','weight_AB'], 
        ascending=[False, False, False])

In [358]:
ventasxpairs_final

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,COD_ARTICULO_B,ARTICULO_B,NRO_FACTURAS_A,weight_A,NRO_FACTURAS_B,weight_B,NRO_FACTURAS_AB,weight_AB
2644951,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,ANTIGRIPALES EXC.ANTIINFARINGITIS,10181,ANTIFLUDES X 100 CAP,1024,2.062500e-04,26094,0.005256,29,5.840939e-06
2644751,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,1024,2.062500e-04,46917,0.009450,19,3.826822e-06
2644775,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,ETICOS AGUDOS,HORMONAS,CORTICOSTEROIDES VIA GENERAL(CORTICOIDES),CORTICOSTEROIDES SOLOS,751140,DEXAMETASONA 8MG IM-IV X 100 AMP/2ML GENERICO LI,1024,2.062500e-04,14677,0.002956,14,2.819764e-06
2644911,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,GASTRICO,APARATO DIGESTIVO Y METABOLICO,DIGESTIVOS INCL.ENZIMAS,DIGESTIVOS INCL.ENZIMAS,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,1024,2.062500e-04,45365,0.009137,13,2.618352e-06
2644956,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,ANTIGRIPALES EXC.ANTIINFARINGITIS,7770108263746,FLAVICOLD PLUS NOCHE X 60 SOB CALIENTE S/ LIMO...,1024,2.062500e-04,17137,0.003452,13,2.618352e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2880878,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,VITAMINAS Y MINERALES,SUPLEMENTOS,SUPLEMENTOS CON MAGNESIO,SUPLEMENTOS CON MAGNESIO,8436000682069,LA JUSTICIA MAGNESIO X 100 COMP TOTAL 5,141,2.840000e-05,1143,0.000230,1,2.014117e-07
2880879,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,VITAMINAS Y MINERALES,SUPLEMENTOS NUTRICIONALES,SUPLEMENTOS DIETARIO ALIMENTICIO,SUPLEMENTOS DIETARIO ALIMENTICIO,7401156632115,ELONGAL X 30 SOBRES MONODOSIS CITRICO,141,2.840000e-05,481,0.000097,1,2.014117e-07
2880880,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,VITAMINAS Y MINERALES,VITAMINAS,VITAMINA C + ASOCIACIONES SIMPLES,VITAMINA C + ASOCIACIONES SIMPLES,7770101008108,C-VIMIN ZINC X 60 SOBRES EFERV VITAMINA C/ZINC,141,2.840000e-05,2845,0.000573,1,2.014117e-07
2880881,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,VITAMINAS Y MINERALES,VITAMINAS,VITAMINA E SOLA O CON ASOCIACIONES,VITAMINA E SOLA O CON ASOCIACIONES,125012,E VIMIN 1000UI X 30 CAP BLANDAS VITAMINA E,141,2.840000e-05,5542,0.001116,1,2.014117e-07


In [359]:
top_sku_pairs = ventasxpairs_final.groupby('COD_ARTICULO_A').head(10).reset_index(drop=True)

In [360]:
top_sku_pairs

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,COD_ARTICULO_B,ARTICULO_B,NRO_FACTURAS_A,weight_A,NRO_FACTURAS_B,weight_B,NRO_FACTURAS_AB,weight_AB
0,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,ANTIGRIPALES EXC.ANTIINFARINGITIS,10181,ANTIFLUDES X 100 CAP,1024,2.062500e-04,26094,0.005256,29,5.840939e-06
1,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,1024,2.062500e-04,46917,0.009450,19,3.826822e-06
2,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,ETICOS AGUDOS,HORMONAS,CORTICOSTEROIDES VIA GENERAL(CORTICOIDES),CORTICOSTEROIDES SOLOS,751140,DEXAMETASONA 8MG IM-IV X 100 AMP/2ML GENERICO LI,1024,2.062500e-04,14677,0.002956,14,2.819764e-06
3,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,GASTRICO,APARATO DIGESTIVO Y METABOLICO,DIGESTIVOS INCL.ENZIMAS,DIGESTIVOS INCL.ENZIMAS,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,1024,2.062500e-04,45365,0.009137,13,2.618352e-06
4,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,EXPECTORANTES,ZAMBON-00008,FLUIMUCIL 600MG X 20 COMP EFERV ACETILCISTEINA,RESFRIO/DOLOR,APARATO RESPIRATORIO,ANTITUSIGENOS/ANTIGRIPALES,ANTIGRIPALES EXC.ANTIINFARINGITIS,7770108263746,FLAVICOLD PLUS NOCHE X 60 SOB CALIENTE S/ LIMO...,1024,2.062500e-04,17137,0.003452,13,2.618352e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135470,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,141,2.840000e-05,46917,0.009450,2,4.028234e-07
135471,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIRREUMATICOS TOPICOS,ANTIRREUMATICOS TOPICOS,7730698014166,FLOGIATRIN 0.5% GEL X 70GR PIROXICAM,141,2.840000e-05,906,0.000182,2,4.028234e-07
135472,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,ETICOS TRATAMIENTO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIULCEROSOS,7703153023980,ESOFAX 40MG X 30 TAB ESOMEPRAZOL,141,2.840000e-05,545,0.000110,2,4.028234e-07
135473,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,ETICOS TRATAMIENTO,SISTEMA NERVIOSO CENTRAL,PSICOANALEPTICOS,ANTIDEPRESIVOS,192615,FLUOXETINA LCH 20MG X 20 COMP,141,2.840000e-05,5883,0.001185,2,4.028234e-07


### Getting the final base for analysis

In [361]:
base_analysis = pd.merge(
    top_sku_pairs,
    skus_analysis,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='right')

In [362]:
cubo.columns

Index(['CAT 1', 'CAT 2', 'CAT 3', 'CAT 4', 'COD_ARTICULO', 'ARTICULO',
       'Precio Unitario FA', 'CR Unitario'],
      dtype='object')

In [363]:
base = pd.merge(
    base_analysis,
    cubo[['COD_ARTICULO', 'Precio Unitario FA', 'CR Unitario']],
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO',
    how='left'
)

In [364]:
base_cat1 = pd.merge(
    base,
    cubo[['COD_ARTICULO', 'CAT 1']],
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='left'
)

In [365]:
base_cat1

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,...,NRO_FACTURAS_alone,weight_gral,weight_alone,weight_overeach,weight_overall,COD_ARTICULO_y,Precio Unitario FA,CR Unitario,COD_ARTICULO,CAT 1
0,RESFRIO/DOLOR,SISTEMA NERVIOSO CENTRAL,ANALGESICOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,...,29887.0,0.01536,0.01096,0.39182,0.00602,7770108268673,8.41,5.968081,7770101005930,RESFRIO/DOLOR
1,RESFRIO/DOLOR,SISTEMA NERVIOSO CENTRAL,ANALGESICOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,...,29887.0,0.01536,0.01096,0.39182,0.00602,7770108268376,9.31,6.817000,7770101005930,RESFRIO/DOLOR
2,RESFRIO/DOLOR,SISTEMA NERVIOSO CENTRAL,ANALGESICOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,ETICOS AGUDOS,ANTIINFECCIOSOS VIA GENERAL,ANTIBACTERIANOS SISTEMICOS,ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO,...,29887.0,0.01536,0.01096,0.39182,0.00602,7703763189090,2.62,1.941500,7770101005930,RESFRIO/DOLOR
3,RESFRIO/DOLOR,SISTEMA NERVIOSO CENTRAL,ANALGESICOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,AGENTES ANTIREUMATICOS ESPECIFICOS,...,29887.0,0.01536,0.01096,0.39182,0.00602,121622,4.16,3.135000,7770101005930,RESFRIO/DOLOR
4,RESFRIO/DOLOR,SISTEMA NERVIOSO CENTRAL,ANALGESICOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,...,29887.0,0.01536,0.01096,0.39182,0.00602,7793640215523,5.33,3.950000,7770101005930,RESFRIO/DOLOR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4951,ETICOS TRATAMIENTO,SISTEMA NERVIOSO CENTRAL,PSICOLEPTICOS,HIPNOTICOS Y SEDANTES,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),ETICOS CRONICOS,SISTEMA NERVIOSO CENTRAL,ANTIEPILEPTICOS,ANTIEPILEPTICOS,...,1130.0,0.00027,0.00041,0.82784,0.00023,7800060412897,3.25,2.125322,7770102002921,ETICOS TRATAMIENTO
4952,ETICOS TRATAMIENTO,SISTEMA NERVIOSO CENTRAL,PSICOLEPTICOS,HIPNOTICOS Y SEDANTES,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),ETICOS AGUDOS,SISTEMA NERVIOSO CENTRAL,PSICOLEPTICOS,TRANQUILIZANTES,...,1130.0,0.00027,0.00041,0.82784,0.00023,7800026007471,11.89,8.666700,7770102002921,ETICOS TRATAMIENTO
4953,ETICOS TRATAMIENTO,SISTEMA NERVIOSO CENTRAL,PSICOLEPTICOS,HIPNOTICOS Y SEDANTES,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),ETICOS TRATAMIENTO,SISTEMA NERVIOSO CENTRAL,PSICOANALEPTICOS,ANTIDEPRESIVOS,...,1130.0,0.00027,0.00041,0.82784,0.00023,7703763170180,0.90,0.591529,7770102002921,ETICOS TRATAMIENTO
4954,ETICOS TRATAMIENTO,SISTEMA NERVIOSO CENTRAL,PSICOLEPTICOS,HIPNOTICOS Y SEDANTES,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),ETICOS TRATAMIENTO,SISTEMA NERVIOSO CENTRAL,PSICOANALEPTICOS,ANTIDEPRESIVOS,...,1130.0,0.00027,0.00041,0.82784,0.00023,7770102002792,19.78,14.614300,7770102002921,ETICOS TRATAMIENTO


In [367]:
base_cat1[['CAT 4_A', 'CAT 4_B']].value_counts()

CAT 4_A                                   CAT 4_B                                    
ANTIRREUMATICOS NO ESTEROIDEOS            ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.       63
                                          ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO    56
ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.  ANTIRREUMATICOS NO ESTEROIDEOS                 54
GASEOSAS                                  AGUA SIN GAS                                   53
ANTIGRIPALES EXC.ANTIINFARINGITIS         ANTIRREUMATICOS NO ESTEROIDEOS                 47
                                                                                         ..
DESCONGESTIONANTES FARINGITICOS           ANTIHISTAMINICOS-ANTIALERGICOS                  1
VITAMINA C + ASOCIACIONES SIMPLES         SUPLEMENTOS CON MAGNESIO                        1
                                          COMPLEJO B AMPOLLA                              1
                                          ANTIRREUMATICOS NO ESTEROIDEOS              

In [240]:
base_final = base_cat1[[
    'CAT 1',
    'COD_ARTICULO_A', 'ARTICULO_A', 
    'COD_ARTICULO_B', 'ARTICULO_B',
    
    'NRO_FACTURAS_A', 'weight_A',
    'NRO_FACTURAS_B', 'weight_B',
    'NRO_FACTURAS_AB', 'weight_AB',
    
    'NRO_FACTURAS_alone', 'weight_alone', 
    'weight_overeach', 'weight_overall',
    
    'Precio Unitario FA',
    'CR Unitario'
]]

In [315]:
base_final = base_final.rename(columns={
    'Precio Unitario FA': 'PVU_B',
    'CR Unitario' : 'CRU_B'
    })

In [316]:
base_final['Profit_B'] = base_final['PVU_B'] - base_final['CRU_B']

In [322]:
base_final.head(10)

,CAT 1,COD_ARTICULO_A,ARTICULO_A,COD_ARTICULO_B,ARTICULO_B,NRO_FACTURAS_A,weight_A,NRO_FACTURAS_B,weight_B,NRO_FACTURAS_AB,...,weight_overeach,weight_overall,PVU_B,CRU_B,Profit_B,Profit_B_avg_x,ambition,opportunity,opportunity_avg,Profit_B_avg_y
0,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7770108268673,NOVADOL 75/500MG X 120 CAP DICLOFENACO/PARACET...,76278.0,0.022211,30450.0,0.008866,1803.0,...,0.4406,0.00979,8.41,5.968081,2.441920,1.716712,0.05,4103.401528,2884.762946,1.716712
1,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7770108268376,NOVADOL FORTE X 30 COMP DICLOFENACO/PARACETAMOL,76278.0,0.022211,25459.0,0.007413,1379.0,...,0.4406,0.00979,9.31,6.817000,2.493000,1.716712,0.05,4189.237200,2884.762946,1.716712
2,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7703763189090,AMOXICILINA 1GR X 20 TAB (LA SANTE),76278.0,0.022211,20531.0,0.005978,1251.0,...,0.4406,0.00979,2.62,1.941500,0.678500,1.716712,0.05,1140.151400,2884.762946,1.716712
3,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,121622,QUETOROL 20MG X 10 TAB KETOROLACO,76278.0,0.022211,16142.0,0.004700,1224.0,...,0.4406,0.00979,4.16,3.135000,1.025000,1.716712,0.05,1722.410000,2884.762946,1.716712
4,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,76278.0,0.022211,46917.0,0.013661,1179.0,...,0.4406,0.00979,5.33,3.861471,1.468529,1.716712,0.05,2467.715627,2884.762946,1.716712
5,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,255701,MIGRANOL X 100 COMP V+,76278.0,0.022211,44849.0,0.013059,995.0,...,0.4406,0.00979,9.18,6.680598,2.499402,1.716712,0.05,4199.995625,2884.762946,1.716712
6,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7750215030448,DOLO FEBREX 800MG X 50 TAB IBUPROFENO FARMACORP,76278.0,0.022211,30324.0,0.008830,968.0,...,0.4406,0.00979,1.99,0.950678,1.039322,1.716712,0.05,1746.476353,2884.762946,1.716712
7,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,38719,NOVADOL 50/500MG X 200 CAP DICLOFENACO/PARACET...,76278.0,0.022211,18339.0,0.005340,719.0,...,0.4406,0.00979,5.74,3.961770,1.778230,1.716712,0.05,2988.137524,2884.762946,1.716712
8,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,76278.0,0.022211,45365.0,0.013209,689.0,...,0.4406,0.00979,5.20,3.794874,1.405126,1.716712,0.05,2361.174403,2884.762946,1.716712
9,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7770102001566,ESPASMO DIOXADOL PLUS X100 COMP PROPIN/CL LISI...,76278.0,0.022211,32299.0,0.009405,674.0,...,0.4406,0.00979,8.01,5.671908,2.338092,1.716712,0.05,3928.929797,2884.762946,1.716712


In [323]:
profit_top_sku = (
    base_final
    .groupby('COD_ARTICULO_A', as_index=False)['Profit_B']
    .mean()
    .rename(columns={'Profit_B': 'Profit_B_avg'})
)

In [324]:
base_final = pd.merge(
    base_final,
    profit_top_sku,
    on='COD_ARTICULO_A'
)

In [325]:
base_final['ambition'] = 0.05

In [326]:
base_final['opportunity'] = base_final['ambition'] * base_final['NRO_FACTURAS_alone'] * base_final['Profit_B']

In [327]:
base_final['opportunity_avg'] = base_final['ambition'] * base_final['NRO_FACTURAS_alone'] * base_final['Profit_B_avg']

In [328]:
base_final

,CAT 1,COD_ARTICULO_A,ARTICULO_A,COD_ARTICULO_B,ARTICULO_B,NRO_FACTURAS_A,weight_A,NRO_FACTURAS_B,weight_B,NRO_FACTURAS_AB,...,weight_overall,PVU_B,CRU_B,Profit_B,Profit_B_avg_x,ambition,opportunity,opportunity_avg,Profit_B_avg_y,Profit_B_avg
0,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7770108268673,NOVADOL 75/500MG X 120 CAP DICLOFENACO/PARACET...,76278.0,0.022211,30450.0,0.008866,1803.0,...,0.00979,8.41,5.968081,2.441920,1.716712,0.05,4103.401528,2884.762946,1.716712,1.716712
1,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7770108268376,NOVADOL FORTE X 30 COMP DICLOFENACO/PARACETAMOL,76278.0,0.022211,25459.0,0.007413,1379.0,...,0.00979,9.31,6.817000,2.493000,1.716712,0.05,4189.237200,2884.762946,1.716712,1.716712
2,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7703763189090,AMOXICILINA 1GR X 20 TAB (LA SANTE),76278.0,0.022211,20531.0,0.005978,1251.0,...,0.00979,2.62,1.941500,0.678500,1.716712,0.05,1140.151400,2884.762946,1.716712,1.716712
3,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,121622,QUETOROL 20MG X 10 TAB KETOROLACO,76278.0,0.022211,16142.0,0.004700,1224.0,...,0.00979,4.16,3.135000,1.025000,1.716712,0.05,1722.410000,2884.762946,1.716712,1.716712
4,RESFRIO/DOLOR,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,76278.0,0.022211,46917.0,0.013661,1179.0,...,0.00979,5.33,3.861471,1.468529,1.716712,0.05,2467.715627,2884.762946,1.716712,1.716712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4053,ETICOS TRATAMIENTO,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),255670,NEURYL 2.5MG GOTAS X 20ML CLONAZEPAM (PSICO),1365.0,0.000397,1412.0,0.000411,4.0,...,0.00034,168.00,123.980000,44.020000,6.395936,0.05,2533.351000,368.086096,6.395936,6.395936
4054,ETICOS TRATAMIENTO,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),7800060412897,CLONEX_CD 0.5MG X 30 COMP DISP CLONAZEPAN (PSICO),1365.0,0.000397,431.0,0.000125,4.0,...,0.00034,3.25,2.125322,1.124678,6.395936,0.05,64.725230,368.086096,6.395936,6.395936
4055,ETICOS TRATAMIENTO,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),7800026007471,ANSIOPLAN_ODT 10MGX30 COMP DISP CLOTIAZEPAM(PS...,1365.0,0.000397,90.0,0.000026,3.0,...,0.00034,11.89,8.666700,3.223300,6.395936,0.05,185.500915,368.086096,6.395936,6.395936
4056,ETICOS TRATAMIENTO,7770102002921,NOCTE_SUBLINGUAL 10MG X 10 COMP ZOLPIDEM (PSICO),7703763170180,AMITRIPTILINA 25MG X 30 TAB (LA SANTE),1365.0,0.000397,4962.0,0.001445,3.0,...,0.00034,0.90,0.591529,0.308471,6.395936,0.05,17.752523,368.086096,6.395936,6.395936


In [250]:
base_final.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\base_analysis_sku.xlsx",
                     index=False)

In [259]:
# Top 50 ARTICULO_A by NRO_FACTURAS_A
if 'NRO_FACTURAS_A' not in base.columns:
    raise KeyError("'NRO_FACTURAS_A' not found in `base`. Check previous merges.")

group_cols = ['COD_ARTICULO_A', 'ARTICULO_A']

top50_articulo_A = (
    base
    .groupby(group_cols, as_index=False)['NRO_FACTURAS_A']
    .max()
    .sort_values(by='NRO_FACTURAS_A', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# attach weight_A if available
if 'weight_A' in base.columns:
    top50_articulo_A = top50_articulo_A.merge(
        base.groupby(group_cols, as_index=False)['weight_A'].max(),
        on=group_cols,
        how='left'
    )


In [258]:

# display
top50_articulo_A

,COD_ARTICULO_A,ARTICULO_A,NRO_FACTURAS_A,weight_A
0,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,76278.0,0.022211
1,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),55343.0,0.016115
2,7770101006746,SUPERAL DIGEST X 100 SOBRES SAL DE FRUTAS,52663.0,0.015334
3,250937,REFRIANEX X 500 COMP (ANTIGRIPAL) V+,49063.0,0.014286
4,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,46917.0,0.013661
5,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,45365.0,0.013209
6,255701,MIGRANOL X 100 COMP V+,44849.0,0.013059
7,751353,VITAMINA C MULTISABOR 60MG X 320 COMP GENERICO LI,38272.0,0.011144
8,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,34636.0,0.010085
9,210656,VIADIL C NF X 10 COMP PROPINOXATO/CL. LISINA,33423.0,0.009732


In [260]:
# Top 50 ARTICULO_B by NRO_FACTURAS_B (including partners count)
if 'NRO_FACTURAS_B' not in base.columns:
    raise KeyError("'NRO_FACTURAS_B' not found in `base`. Check previous merges.")

group_cols_b = ['COD_ARTICULO_B', 'ARTICULO_B']

# aggregate invoices per ARTICULO_B
agg_b = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(NRO_FACTURAS_B=('NRO_FACTURAS_B', 'max'))
)

# count distinct partners (COD_ARTICULO_A) per ARTICULO_B
partners = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(partners_count=('COD_ARTICULO_A', 'nunique'))
)

# merge results
top50_articulo_B = agg_b.merge(partners, on=group_cols_b, how='left')

# attach weight_B if available
if 'weight_B' in base.columns:
    w = base.groupby(group_cols_b, as_index=False)['weight_B'].max()
    top50_articulo_B = top50_articulo_B.merge(w, on=group_cols_b, how='left')

# sort and take top 50
top50_articulo_B = (
    top50_articulo_B
    .sort_values(by='NRO_FACTURAS_B', ascending=False)
    .head(100)
    .reset_index(drop=True)
)


In [262]:
# display
top50_articulo_B

,COD_ARTICULO_B,ARTICULO_B,NRO_FACTURAS_B,partners_count,weight_B
0,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,76278.0,47,0.022211
1,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),55343.0,10,0.016115
2,7770101006746,SUPERAL DIGEST X 100 SOBRES SAL DE FRUTAS,52663.0,40,0.015334
3,250937,REFRIANEX X 500 COMP (ANTIGRIPAL) V+,49063.0,25,0.014286
4,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,46917.0,281,0.013661
...,...,...,...,...,...
95,7798032938424,ALMUXIM 100MG X 2 COMP SILDENAFIL,8697.0,17,0.002532
96,7803510002549,CIRUELAX FORTE X 100 COMP CIRUELA,8585.0,6,0.002500
97,25616,BIOTICO 500MG X 5 COMP AZITROMICINA,8515.0,18,0.002479
98,7750215030356,DICLOFENACO 100MG X 100 TAB LIB PROL FARMACORP,8443.0,4,0.002458
